# Miniproyecto: Trayectoria interestelar de 1I/'Oumuamua con `pymcel`

Objetivo: estimar el vector de procedencia de 1I/'Oumuamua usando efemérides reales, caracterización orbital hiperbólica y reconstrucción geométrica en 3D.

## 1) Setup

Convenciones usadas (alineadas con `agents.md`):
- `consulta_horizons()` para efemérides reales.
- Estado en **km y km/s**.
- Parámetro gravitacional solar tomado desde `pymcel.constantes`.

In [ ]:
import numpy as np
import pandas as pd
import pymcel as pc
from pymcel import constantes as const
import plotly.graph_objects as go

In [ ]:
def _to_float(x):
    return float(x.value) if hasattr(x, 'value') else float(x)

# Preferimos mu_sun (SPICE, típicamente km^3/s^2); fallback a GM_sun.
MU_SUN_RAW = getattr(const, 'mu_sun', getattr(const, 'GM_sun'))
MU_SUN = _to_float(MU_SUN_RAW)

# Si viene en SI (m^3/s^2), convertir a km^3/s^2.
if MU_SUN > 1e15:
    MU_SUN = MU_SUN / 1e9

print(f"mu_sun usado = {MU_SUN:.6e} km^3/s^2")

## 2) Fase 1: Obtención del vector de estado real

In [ ]:
# Época cercana al perihelio de 1I para obtener un estado representativo.
epoch = '2017-10-14'

datos = pc.consulta_horizons(
    id='1I',
    location='@0',
    epochs=epoch,
    datos='vectors',
    propiedades=[
        ('x', 'km'), ('y', 'km'), ('z', 'km'),
        ('vx', 'km/s'), ('vy', 'km/s'), ('vz', 'km/s')
    ]
)

df = pd.DataFrame(datos)
print(df.head(1))

In [ ]:
def extrae_estado_km(df):
    # Mapeo robusto de columnas (por si Horizons retorna variantes de nombre).
    norm = {c.lower().replace('_', ''): c for c in df.columns}
    keys = ['x', 'y', 'z', 'vx', 'vy', 'vz']
    vals = []
    for k in keys:
        if k not in norm:
            raise KeyError(f"No se encontró columna '{k}' en {list(df.columns)}")
        vals.append(float(df.iloc[0][norm[k]]))
    return np.array(vals, dtype=float)

estado = extrae_estado_km(df)
r0 = estado[:3]
v0 = estado[3:]

print('r0 [km]   =', r0)
print('v0 [km/s] =', v0)

## 3) Fase 2: Caracterización de la hipérbola

Se obtienen elementos orbitales con `estado_a_elementos(mu, estado)` y se valida energía específica positiva.

In [ ]:
# elementos = (p, e, i, Omega, omega, f) según convención de pymcel.
p, e, inc, Omega, omega, f = pc.estado_a_elementos(MU_SUN, estado)

energia_especifica = 0.5 * np.dot(v0, v0) - MU_SUN / np.linalg.norm(r0)

print(f"p = {p}")
print(f"e = {e}")
print(f"i = {inc} deg")
print(f"Omega = {Omega} deg")
print(f"omega = {omega} deg")
print(f"f = {f} deg")
print(f"Energía específica ε = {energia_especifica:.6f} km^2/s^2")
print('¿Órbita hiperbólica (ε > 0)?', energia_especifica > 0)

## 4) Fase 3: Reconstrucción forense (asíntota de entrada)

Para hipérbola: \(\cos\psi = 1/e\). Con ello se obtiene la dirección asintótica en el plano perifocal y luego se rota al marco inercial con Euler \(\mathcal{M}(\omega, i, \Omega)\).

In [ ]:
def R1(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def R3(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def perifocal_a_inercial(vec_pf, omega_deg, inc_deg, Omega_deg):
    w = np.deg2rad(omega_deg)
    i = np.deg2rad(inc_deg)
    O = np.deg2rad(Omega_deg)
    M = R3(O) @ R1(i) @ R3(w)
    return M @ vec_pf

if e <= 1:
    raise ValueError('La órbita no es hiperbólica (e <= 1), no hay asíntota real.')

psi = np.arccos(1.0 / e)
f_inf = np.arccos(-1.0 / e)    # |f| al infinito
f_in = -f_inf                  # rama de entrada
u_in_pf = np.array([np.cos(f_in), np.sin(f_in), 0.0])
u_in_eci = perifocal_a_inercial(u_in_pf, omega, inc, Omega)
u_in_eci = u_in_eci / np.linalg.norm(u_in_eci)

ra = (np.degrees(np.arctan2(u_in_eci[1], u_in_eci[0])) + 360) % 360
dec = np.degrees(np.arcsin(u_in_eci[2]))

print(f"psi = {np.degrees(psi):.6f} deg")
print('Vector unitario de procedencia (ECI):', u_in_eci)
print(f"RA = {ra:.6f} deg, Dec = {dec:.6f} deg")

## 5) Fase 4: Visualización 3D de la trayectoria de escape

Se integra una trayectoria de dos cuerpos Sol-objeto y se visualiza con `plot_ncuerpos_3d(tipo='plotly')`. Se resalta en rojo el vector de procedencia interestelar.

In [ ]:
# Integración simétrica alrededor de la época de consulta (~2 años).
ts = np.linspace(-2 * 365.25 * 24 * 3600, 2 * 365.25 * 24 * 3600, 1200)

rs_obj, vs_obj = pc.doscuerpos_solucion(MU_SUN, r0, v0, ts)

rs_obj = np.asarray(rs_obj)
vs_obj = np.asarray(vs_obj)

if rs_obj.ndim == 3 and rs_obj.shape[0] == 1:
    rs_obj = rs_obj[0]
if vs_obj.ndim == 3 and vs_obj.shape[0] == 1:
    vs_obj = vs_obj[0]

if rs_obj.ndim != 2 or rs_obj.shape[1] != 3:
    raise ValueError(f"Forma inesperada de rs_obj: {rs_obj.shape}")

sun_rs = np.zeros_like(rs_obj)
sun_vs = np.zeros_like(vs_obj)

# Formato N-cuerpos: [N, Nt, 3]
rs_plot = np.stack([sun_rs, rs_obj], axis=0)
vs_plot = np.stack([sun_vs, vs_obj], axis=0)

fig = pc.plot_ncuerpos_3d(rs_plot, vs_plot, tipo='plotly')

L = 1.2 * np.max(np.linalg.norm(rs_obj, axis=1))
fig.add_trace(go.Scatter3d(
    x=[0.0, L * u_in_eci[0]],
    y=[0.0, L * u_in_eci[1]],
    z=[0.0, L * u_in_eci[2]],
    mode='lines+markers',
    name='Dirección de procedencia interestelar',
    line=dict(color='crimson', width=7),
    marker=dict(size=4, color='crimson')
))

fig.update_layout(title="1I/'Oumuamua: trayectoria hiperbólica y vector de procedencia")
fig.show()

## 6) Hipótesis de procedencia (siguiente paso científico)

Con el vector inercial de llegada (RA/Dec) puedes:
1. Transformarlo a marco galáctico.
2. Cruce con catálogos estelares (Gaia) para buscar sistemas con velocidad relativa compatible.
3. Propagar incertidumbres variando época y estado inicial.